# Notebook 1: Data loading and exploratory analysis

**Project:** Cross-Domain Spam Detection  
**Research question:** Can a spam classifier trained on SMS messages generalize to emails?

This notebook loads two independent Hugging Face datasets, standardizes their schema, creates reproducible splits, and verifies that the data is leakage-free before modeling. It runs in VS Code or Kaggle.

## Datasets and decisions

- [jngb-labs/sms-spam](https://huggingface.co/datasets/jngb-labs/sms-spam): short SMS messages.
- [SetFit/enron_spam](https://huggingface.co/datasets/SetFit/enron_spam): longer email messages.

Both datasets are standardized to `text`, `label`, and `source`, with `0 = ham` and `1 = spam`. Cleaning normalizes whitespace and removes empty texts, case-insensitive exact duplicates, and label conflicts. Punctuation, capitalization, URLs, and numbers are retained as possible spam signals.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / 'src').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import LABEL_NAMES, load_prepared_splits, summarize_splits
from src.protocol import DATA_SPLIT_SEED

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)
print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')

## 1. Load, clean, and split

SMS is split 70/15/15 with stratification. For Enron, the cleaned published test partition remains the test set; training rows that duplicate it are removed before 15% of the remaining training data is reserved for validation.

The resulting partitions are frozen with split seed `42` and reused in every notebook. Later training seeds change model initialization and training order, never which examples belong to train, validation, or test.

In [ ]:
splits, cleaning_audit = load_prepared_splits(random_state=DATA_SPLIT_SEED)
cleaning_audit

In [ ]:
split_summary = summarize_splits(splits)
display_summary = split_summary.copy()
display_summary['spam_rate'] = display_summary['spam_rate'].map(lambda value: f'{value:.1%}')
display_summary['median_characters'] = display_summary['median_characters'].round().astype(int)
display_summary['p95_characters'] = display_summary['p95_characters'].round().astype(int)
display_summary

## 2. Shared EDA frame

All splits are combined for EDA while retaining `source` and `split`. This supports domain comparisons without losing split identity; modeling still uses the original split dictionaries.

In [ ]:
eda = pd.concat(
    [
        frame.assign(dataset=dataset_name, split=split_name)
        for dataset_name, dataset_splits in splits.items()
        for split_name, frame in dataset_splits.items()
    ],
    ignore_index=True,
)
eda['label_name'] = eda['label'].map(LABEL_NAMES)
eda['characters'] = eda['text'].str.len()
eda['words'] = eda['text'].str.split().str.len()


## 3. Training class balance

Because SMS is imbalanced, accuracy alone is not sufficient. In the robustness extension, validation macro-F1 ranks candidate configurations so both classes affect model selection. Spam-class F1 (`spam = 1`) remains the headline test metric for the primary SMS -> Enron transfer direction, supported by precision, recall, ROC-AUC, and confusion matrices.

In [ ]:
train_eda = eda.query("split == 'train'").copy()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, dataset_name in zip(axes, ['sms', 'enron']):
    counts = (
        train_eda.query('dataset == @dataset_name')['label_name']
        .value_counts()
        .reindex(['ham', 'spam'])
    )
    sns.barplot(x=counts.index, y=counts.values, hue=counts.index, legend=False, ax=axis)
    axis.set_title(f'{dataset_name.upper()} training labels')
    axis.set_xlabel('Class')
    axis.set_ylabel('Messages')
    for container in axis.containers:
        axis.bar_label(container, fmt='%d')

plt.tight_layout()
plt.show()

### Exact counts and count-matched control

The frozen training splits contain 3,162 ham and 449 spam SMS messages, versus 12,411 ham and 11,521 spam Enron emails. These counts define the Enron count-matched control in Notebook 4: exactly 3,162 ham and 449 spam emails are sampled from the Enron training split, without replacement, while Enron validation and test remain unchanged.

In [ ]:
training_class_counts = (
    train_eda.groupby(['dataset', 'label_name'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=['sms', 'enron'], columns=['ham', 'spam'])
    .astype(int)
)
count_matched_enron_train = pd.DataFrame(
    {
        'enron_train_available': training_class_counts.loc['enron'],
        'target_equal_to_sms_train': training_class_counts.loc['sms'],
    }
).astype(int)

display(training_class_counts.rename_axis(columns='class'))
count_matched_enron_train.rename_axis('class')

## 4. Message-length domain shift

SMS and email have substantially different length distributions. The logarithmic scale makes both visible and highlights a domain shift that may affect transfer performance.

In [ ]:
length_summary = (
    train_eda.groupby(['dataset', 'label_name'])[['characters', 'words']]
    .agg(['median', lambda values: values.quantile(0.95)])
)
length_summary.columns = [
    '_'.join([left, 'median' if right == 'median' else 'p95'])
    for left, right in length_summary.columns
]
length_summary.round(0)

In [ ]:
plot_data = train_eda.copy()
plot_data['log10_characters'] = np.log10(plot_data['characters'] + 1)

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=plot_data,
    x='dataset',
    y='log10_characters',
    hue='label_name',
    showfliers=False,
)
plt.title('Training message lengths by domain and class')
plt.xlabel('Dataset')
plt.ylabel('log10(characters + 1)')
plt.legend(title='Label')
plt.tight_layout()
plt.show()

### DistilBERT token-length analysis

Token lengths are computed for every prepared train, validation, and test row with `distilbert-base-uncased`. Counts are tokenizer-specific, include special tokens, and use neither truncation nor padding. For each predeclared maximum length, the reported rate is the percentage of rows whose untruncated count exceeds the limit and would therefore be truncated. The table is descriptive and does not select a configuration from test data.

In [ ]:
from transformers import AutoTokenizer

DISTILBERT_TOKENIZER = 'distilbert-base-uncased'
TOKENIZATION_BATCH_SIZE = 256

tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_TOKENIZER, use_fast=True)
token_counts = []

for start in range(0, len(eda), TOKENIZATION_BATCH_SIZE):
    batch_texts = eda['text'].iloc[start:start + TOKENIZATION_BATCH_SIZE].tolist()
    encoded = tokenizer(
        batch_texts,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_length=True,
        verbose=False,
    )
    token_counts.extend(encoded['length'])

eda['distilbert_tokens'] = np.asarray(token_counts, dtype=np.int32)
assert len(eda['distilbert_tokens']) == len(eda)

In [ ]:
token_length_summary = (
    eda.groupby(['dataset', 'label_name'], observed=True)['distilbert_tokens']
    .agg(
        rows='size',
        median='median',
        p90=lambda values: values.quantile(0.90),
        p95=lambda values: values.quantile(0.95),
    )
    .round(1)
)
token_length_summary

In [ ]:
TOKEN_LIMITS = (64, 128, 256, 512)
truncation_rates = (
    eda.groupby(['dataset', 'label_name'], observed=True)['distilbert_tokens']
    .agg(
        **{
            f'max_{limit}_pct': (
                lambda values, max_length=limit: values.gt(max_length).mean() * 100
            )
            for limit in TOKEN_LIMITS
        }
    )
    .round(1)
)
truncation_rates

## 5. Representative examples

Only short previews are displayed to keep the notebook readable.

In [ ]:
examples = (
    train_eda.groupby(['dataset', 'label_name'], group_keys=False)
    .sample(n=2, random_state=DATA_SPLIT_SEED)
    .loc[:, ['dataset', 'label_name', 'text']]
)
examples['text_preview'] = examples['text'].str.slice(0, 240)
examples.drop(columns='text').sort_values(['dataset', 'label_name'])

## 6. Cross-source overlap check

Exact overlap between the sources could inflate cross-domain performance. We compare normalized texts and confirm that SMS and Enron share no exact messages.

In [ ]:
sms_keys = set(eda.query("dataset == 'sms'")['text'].str.casefold())
enron_keys = set(eda.query("dataset == 'enron'")['text'].str.casefold())
cross_source_overlap = sms_keys & enron_keys
print(f'Exact normalized texts shared by SMS and Enron: {len(cross_source_overlap):,}')

## Conclusion

- SMS contains 5,159 unique messages, with about 12.4% spam.
- The cleaned Enron training partition contains 28,528 unique messages; 372 messages also found in the cleaned test partition are excluded before splitting.
- The cleaned Enron test partition contains 1,981 messages and remains held out.
- Median training length is 61 characters for SMS and 710 for Enron.
- No case-insensitive exact text is shared between the final SMS and Enron data.

The data presents two immediate modeling challenges: SMS class imbalance and a large document-length shift. The exact training counts and tokenizer-specific summaries define the count-matched and maximum-length controls in Notebook 4. Notebook 2 evaluates a TF-IDF logistic-regression baseline on the same frozen splits.